<a href="https://www.kaggle.com/code/cythara/project-randomised-scaling-techniques?scriptVersionId=212374598" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# CODE FOR RANDOMISED SCALING TECHNIQUES...

# Processing for first 10K entries.

In [2]:
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.random_projection import GaussianRandomProjection
from sklearn.random_projection import johnson_lindenstrauss_min_dim
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Function to record time
def record_time(start_time):
    return time.time() - start_time

# Function to process data in chunks with optimized printing
def process_in_chunks(data, func, chunk_size=1_000, print_every=100_000):
    result = []
    total_time = 0
    total_rows_processed = 0
    
    for start in range(0, data.shape[0], chunk_size):
        end = min(start + chunk_size, data.shape[0])
        chunk = data[start:end]
        chunk_start_time = time.time()
        transformed_chunk = func(chunk)
        chunk_time = record_time(chunk_start_time)
        total_time += chunk_time
        result.append(transformed_chunk)
        
        # Print progress every 'print_every' rows
        total_rows_processed += chunk_size
        if total_rows_processed % print_every == 0 or total_rows_processed >= data.shape[0]:
            print(f"Processed {total_rows_processed} rows so far - Last batch shape: {transformed_chunk.shape} - Time taken: {chunk_time:.2f}s")
    
    print(f"Total time for processing chunks: {total_time:.2f}s")
    return np.vstack(result), total_time

# Randomized Feature Expansion in Chunks
def random_projection(chunk):
    grp = GaussianRandomProjection(n_components=target_feature_count, random_state=42)
    return grp.fit_transform(chunk)

# Load the dataset
start_time = time.time()
print("\nStep 1: Loading dataset")
data = pd.read_csv("/kaggle/input/brewery-operations-and-market-analysis-dataset/brewery_data_complete_extended.csv")
data = data.sample(frac=0.001, random_state=42)
loading_time = record_time(start_time)
print(f"Dataset loaded. Time taken: {loading_time:.2f}s")

# Drop the 'Brew_Date' column
print("\nStep 2: Preprocessing dataset")
data = data.drop(columns=['Brew_Date'])
categorical_columns = data.select_dtypes(include=['object', 'category']).columns

# Apply one-hot encoding in chunks
start_time = time.time()
data_encoded = one_hot_encode_in_chunks(data, categorical_columns, chunk_size=1_000)
encoding_time = record_time(start_time)
print(f"Size of the dataset after one-hot encoding: {data_encoded.shape}")
print(f"Time taken for one-hot encoding: {encoding_time:.2f}s")

data_encoded = data_encoded.fillna(0)

# Split predictors and target
X = data_encoded.drop(columns=['Total_Sales'])
y = data_encoded['Total_Sales']

# Dimensionality Expansion
start_time = time.time()
print("\nStep 3: Expanding feature space using random projection")
target_feature_count = 10 * X.shape[1]
expanded_random, expansion_time = process_in_chunks(X.values, random_projection, chunk_size=1_000, print_every=100_000)
print(f"Size after random projection feature expansion: {expanded_random.shape}")
print(f"Total time for feature expansion: {expansion_time:.2f}s")

# Dimensionality Reduction Using JL Lemma
start_time = time.time()
print("\nStep 4: Reducing dimensionality using JL Lemma")
eps = 0.6  # Larger epsilon allows more dimensionality reduction
reduced_dim = johnson_lindenstrauss_min_dim(n_samples=X.shape[0], eps=eps)
adjusted_reduced_dim = min(reduced_dim, expanded_random.shape[1])
print(f"Reduced dimensionality using JL Lemma with epsilon={eps}: {adjusted_reduced_dim}")

svd_random = TruncatedSVD(n_components=adjusted_reduced_dim, random_state=42)
reduced_random = svd_random.fit_transform(expanded_random)
reduction_time = record_time(start_time)
print(f"Size after JL Lemma reduction: {reduced_random.shape}")
print(f"Time taken for dimensionality reduction: {reduction_time:.2f}s")

# Model Training and Evaluation
start_time = time.time()
print("\nStep 5: Splitting data and training Linear Regression model")
X_train, X_test, y_train, y_test = train_test_split(reduced_random, y[:reduced_random.shape[0]], test_size=0.2, random_state=42)
split_time = record_time(start_time)
print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
print(f"Time taken for train-test split: {split_time:.2f}s")

start_time = time.time()
model = LinearRegression()
model.fit(X_train, y_train)
training_time = record_time(start_time)
print(f"Model training completed. Time taken: {training_time:.2f}s")

# Model evaluation
start_time = time.time()
print("\nStep 6: Evaluating the model")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
evaluation_time = record_time(start_time)

print(f"Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
print(f"Train R²: {train_r2:.2f}, Test R²: {test_r2:.2f}")
print(f"Time taken for evaluation: {evaluation_time:.2f}s")




Step 1: Loading dataset
Dataset loaded. Time taken: 56.29s

Step 2: Preprocessing dataset
Size of the dataset after one-hot encoding: (10000, 688)
Time taken for one-hot encoding: 0.31s

Step 3: Expanding feature space using random projection
Processed 10000 rows so far - Last batch shape: (1000, 6870) - Time taken: 0.44s
Total time for processing chunks: 4.37s
Size after random projection feature expansion: (10000, 6870)
Total time for feature expansion: 4.37s

Step 4: Reducing dimensionality using JL Lemma
Reduced dimensionality using JL Lemma with epsilon=0.6: 341
Size after JL Lemma reduction: (10000, 341)
Time taken for dimensionality reduction: 17.18s

Step 5: Splitting data and training Linear Regression model
Training data shape: (8000, 341), Testing data shape: (2000, 341)
Time taken for train-test split: 0.01s
Model training completed. Time taken: 0.21s

Step 6: Evaluating the model
Train MSE: 28774451.05, Test MSE: 31094240.82
Train R²: 0.04, Test R²: -0.05
Time taken for e

# Processing for 100K entries.

In [3]:
# import pandas as pd
# import numpy as np
# import time
# import warnings
# from sklearn.random_projection import GaussianRandomProjection
# from sklearn.random_projection import johnson_lindenstrauss_min_dim
# from sklearn.decomposition import TruncatedSVD
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, r2_score

# # Suppress warnings
# warnings.filterwarnings("ignore", category=UserWarning)

# # Function to record time
# def record_time(start_time):
#     return time.time() - start_time

# # Function to process data in chunks with optimized printing
# def process_in_chunks(data, func, chunk_size=1_000, print_every=100_000):
#     result = []
#     total_time = 0
#     total_rows_processed = 0
    
#     for start in range(0, data.shape[0], chunk_size):
#         end = min(start + chunk_size, data.shape[0])
#         chunk = data[start:end]
#         chunk_start_time = time.time()
#         transformed_chunk = func(chunk)
#         chunk_time = record_time(chunk_start_time)
#         total_time += chunk_time
#         result.append(transformed_chunk)
        
#         # Print progress every 'print_every' rows
#         total_rows_processed += chunk_size
#         if total_rows_processed % print_every == 0 or total_rows_processed >= data.shape[0]:
#             print(f"Processed {total_rows_processed} rows so far - Last batch shape: {transformed_chunk.shape} - Time taken: {chunk_time:.2f}s")
    
#     print(f"Total time for processing chunks: {total_time:.2f}s")
#     return np.vstack(result), total_time

# # Function for one-hot encoding in chunks
# def one_hot_encode_in_chunks(data, categorical_columns, chunk_size=10_000):
#     encoded_data = []
#     for start in range(0, data.shape[0], chunk_size):
#         end = min(start + chunk_size, data.shape[0])
#         chunk = data.iloc[start:end]
#         encoded_chunk = pd.get_dummies(chunk, columns=categorical_columns, drop_first=False)
#         encoded_data.append(encoded_chunk)
#     return pd.concat(encoded_data, axis=0)

# # Randomized Feature Expansion in Chunks
# def random_projection(chunk):
#     grp = GaussianRandomProjection(n_components=target_feature_count, random_state=42)
#     return grp.fit_transform(chunk)

# # Load the dataset
# start_time = time.time()
# print("\nStep 1: Loading dataset")
# data = pd.read_csv("/kaggle/input/brewery-operations-and-market-analysis-dataset/brewery_data_complete_extended.csv")
# data = data.sample(frac=0.01, random_state=42)
# loading_time = record_time(start_time)
# print(f"Dataset loaded. Time taken: {loading_time:.2f}s")

# # Drop the 'Brew_Date' column
# print("\nStep 2: Preprocessing dataset")
# data = data.drop(columns=['Brew_Date'])
# categorical_columns = data.select_dtypes(include=['object', 'category']).columns

# # Apply one-hot encoding in chunks
# start_time = time.time()
# data_encoded = one_hot_encode_in_chunks(data, categorical_columns, chunk_size=1_000)
# encoding_time = record_time(start_time)
# print(f"Size of the dataset after one-hot encoding: {data_encoded.shape}")
# print(f"Time taken for one-hot encoding: {encoding_time:.2f}s")

# data_encoded = data_encoded.fillna(0)

# # Dimensionality Expansion
# start_time = time.time()
# print("\nStep 3: Expanding feature space using random projection")
# target_feature_count = 10 * data_encoded.shape[1]
# expanded_random, expansion_time = process_in_chunks(data_encoded.values, random_projection, chunk_size=1_000, print_every=100_000)
# print(f"Size after random projection feature expansion: {expanded_random.shape}")
# print(f"Total time for feature expansion: {expansion_time:.2f}s")

# # Dimensionality Reduction Using JL Lemma
# start_time = time.time()
# print("\nStep 4: Reducing dimensionality using JL Lemma")
# eps = 0.6  # Larger epsilon allows more dimensionality reduction
# reduced_dim = johnson_lindenstrauss_min_dim(n_samples=data_encoded.shape[0], eps=eps)
# adjusted_reduced_dim = min(reduced_dim, expanded_random.shape[1])
# print(f"Reduced dimensionality using JL Lemma with epsilon={eps}: {adjusted_reduced_dim}")

# svd_random = TruncatedSVD(n_components=adjusted_reduced_dim, random_state=42)
# reduced_random = svd_random.fit_transform(expanded_random)
# reduction_time = record_time(start_time)
# print(f"Size after JL Lemma reduction: {reduced_random.shape}")
# print(f"Time taken for dimensionality reduction: {reduction_time:.2f}s")

# # Model Training and Evaluation
# start_time = time.time()
# print("\nStep 5: Splitting data and training Linear Regression model")
# # 'Total_Sales' is the target column
# y = data_encoded['Total_Sales'][:reduced_random.shape[0]]
# reduced_random = reduced_random.drop(columns=['Total_Sales'])
# X_train, X_test, y_train, y_test = train_test_split(reduced_random, y, test_size=0.2, random_state=42)
# split_time = record_time(start_time)
# print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
# print(f"Time taken for train-test split: {split_time:.2f}s")

# start_time = time.time()
# model = LinearRegression()
# model.fit(X_train, y_train)
# training_time = record_time(start_time)
# print(f"Model training completed. Time taken: {training_time:.2f}s")

# # Model evaluation
# start_time = time.time()
# print("\nStep 6: Evaluating the model")
# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

# train_mse = mean_squared_error(y_train, y_train_pred)
# test_mse = mean_squared_error(y_test, y_test_pred)

# train_r2 = r2_score(y_train, y_train_pred)
# test_r2 = r2_score(y_test, y_test_pred)
# evaluation_time = record_time(start_time)

# print(f"Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
# print(f"Train R²: {train_r2:.2f}, Test R²: {test_r2:.2f}")
# print(f"Time taken for evaluation: {evaluation_time:.2f}s")

import pandas as pd
import numpy as np
import time
import warnings
from sklearn.random_projection import GaussianRandomProjection
from sklearn.random_projection import johnson_lindenstrauss_min_dim
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Function to record time
def record_time(start_time):
    return time.time() - start_time

# Function to process data in chunks with optimized printing
def process_in_chunks(data, func, chunk_size=1_000, print_every=100_000):
    result = []
    total_time = 0
    total_rows_processed = 0
    
    for start in range(0, data.shape[0], chunk_size):
        end = min(start + chunk_size, data.shape[0])
        chunk = data[start:end]
        chunk_start_time = time.time()
        transformed_chunk = func(chunk)
        chunk_time = record_time(chunk_start_time)
        total_time += chunk_time
        result.append(transformed_chunk)
        
        # Print progress every 'print_every' rows
        total_rows_processed += chunk_size
        if total_rows_processed % print_every == 0 or total_rows_processed >= data.shape[0]:
            print(f"Processed {total_rows_processed} rows so far - Last batch shape: {transformed_chunk.shape} - Time taken: {chunk_time:.2f}s")
    
    print(f"Total time for processing chunks: {total_time:.2f}s")
    return np.vstack(result), total_time

# Randomized Feature Expansion in Chunks
def random_projection(chunk):
    grp = GaussianRandomProjection(n_components=target_feature_count, random_state=42)
    return grp.fit_transform(chunk)

# Load the dataset
start_time = time.time()
print("\nStep 1: Loading dataset")
data = pd.read_csv("/kaggle/input/brewery-operations-and-market-analysis-dataset/brewery_data_complete_extended.csv")
data = data.sample(frac=0.01, random_state=42)
loading_time = record_time(start_time)
print(f"Dataset loaded. Time taken: {loading_time:.2f}s")

# Drop the 'Brew_Date' column
print("\nStep 2: Preprocessing dataset")
data = data.drop(columns=['Brew_Date'])
categorical_columns = data.select_dtypes(include=['object', 'category']).columns

# Apply one-hot encoding in chunks
start_time = time.time()
data_encoded = one_hot_encode_in_chunks(data, categorical_columns, chunk_size=1_000)
encoding_time = record_time(start_time)
print(f"Size of the dataset after one-hot encoding: {data_encoded.shape}")
print(f"Time taken for one-hot encoding: {encoding_time:.2f}s")

data_encoded = data_encoded.fillna(0)

# Split predictors and target
X = data_encoded.drop(columns=['Total_Sales'])
y = data_encoded['Total_Sales']

# Dimensionality Expansion
start_time = time.time()
print("\nStep 3: Expanding feature space using random projection")
target_feature_count = 10 * X.shape[1]
expanded_random, expansion_time = process_in_chunks(X.values, random_projection, chunk_size=1_000, print_every=100_000)
print(f"Size after random projection feature expansion: {expanded_random.shape}")
print(f"Total time for feature expansion: {expansion_time:.2f}s")

# Dimensionality Reduction Using JL Lemma
start_time = time.time()
print("\nStep 4: Reducing dimensionality using JL Lemma")
eps = 0.6  # Larger epsilon allows more dimensionality reduction
reduced_dim = johnson_lindenstrauss_min_dim(n_samples=X.shape[0], eps=eps)
adjusted_reduced_dim = min(reduced_dim, expanded_random.shape[1])
print(f"Reduced dimensionality using JL Lemma with epsilon={eps}: {adjusted_reduced_dim}")

svd_random = TruncatedSVD(n_components=adjusted_reduced_dim, random_state=42)
reduced_random = svd_random.fit_transform(expanded_random)
reduction_time = record_time(start_time)
print(f"Size after JL Lemma reduction: {reduced_random.shape}")
print(f"Time taken for dimensionality reduction: {reduction_time:.2f}s")

# Model Training and Evaluation
start_time = time.time()
print("\nStep 5: Splitting data and training Linear Regression model")
X_train, X_test, y_train, y_test = train_test_split(reduced_random, y[:reduced_random.shape[0]], test_size=0.2, random_state=42)
split_time = record_time(start_time)
print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
print(f"Time taken for train-test split: {split_time:.2f}s")

start_time = time.time()
model = LinearRegression()
model.fit(X_train, y_train)
training_time = record_time(start_time)
print(f"Model training completed. Time taken: {training_time:.2f}s")

# Model evaluation
start_time = time.time()
print("\nStep 6: Evaluating the model")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
evaluation_time = record_time(start_time)

print(f"Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
print(f"Train R²: {train_r2:.2f}, Test R²: {test_r2:.2f}")
print(f"Time taken for evaluation: {evaluation_time:.2f}s")




Step 1: Loading dataset
Dataset loaded. Time taken: 52.29s

Step 2: Preprocessing dataset
Size of the dataset after one-hot encoding: (100000, 688)
Time taken for one-hot encoding: 3.31s

Step 3: Expanding feature space using random projection
Processed 100000 rows so far - Last batch shape: (1000, 6870) - Time taken: 0.47s
Total time for processing chunks: 46.99s
Size after random projection feature expansion: (100000, 6870)
Total time for feature expansion: 46.99s

Step 4: Reducing dimensionality using JL Lemma
Reduced dimensionality using JL Lemma with epsilon=0.6: 426
Size after JL Lemma reduction: (100000, 426)
Time taken for dimensionality reduction: 159.71s

Step 5: Splitting data and training Linear Regression model
Training data shape: (80000, 426), Testing data shape: (20000, 426)
Time taken for train-test split: 0.14s
Model training completed. Time taken: 3.29s

Step 6: Evaluating the model
Train MSE: 29968252.80, Test MSE: 30475722.41
Train R²: 0.01, Test R²: -0.00
Time ta

# Processing for 10M entries

In [ ]:
# import pandas as pd
# import numpy as np
# import time
# import warnings
# from sklearn.random_projection import GaussianRandomProjection
# from sklearn.random_projection import johnson_lindenstrauss_min_dim
# from sklearn.decomposition import TruncatedSVD
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import mean_squared_error, r2_score

# # Suppress warnings
# warnings.filterwarnings("ignore", category=UserWarning)

# # Function to record time
# def record_time(start_time):
#     return time.time() - start_time

# # Function to process data in chunks with optimized printing
# def process_in_chunks(data, func, chunk_size=1_000, print_every=100_000):
#     result = []
#     total_time = 0
#     total_rows_processed = 0
    
#     for start in range(0, data.shape[0], chunk_size):
#         end = min(start + chunk_size, data.shape[0])
#         chunk = data[start:end]
#         chunk_start_time = time.time()
#         transformed_chunk = func(chunk)
#         chunk_time = record_time(chunk_start_time)
#         total_time += chunk_time
#         result.append(transformed_chunk)
        
#         # Print progress every 'print_every' rows
#         total_rows_processed += chunk_size
#         if total_rows_processed % print_every == 0 or total_rows_processed >= data.shape[0]:
#             print(f"Processed {total_rows_processed} rows so far - Last batch shape: {transformed_chunk.shape} - Time taken: {chunk_time:.2f}s")
    
#     print(f"Total time for processing chunks: {total_time:.2f}s")
#     return np.vstack(result), total_time

# # Function for one-hot encoding in chunks
# def one_hot_encode_in_chunks(data, categorical_columns, chunk_size=10_000):
#     encoded_data = []
#     for start in range(0, data.shape[0], chunk_size):
#         end = min(start + chunk_size, data.shape[0])
#         chunk = data.iloc[start:end]
#         encoded_chunk = pd.get_dummies(chunk, columns=categorical_columns, drop_first=False)
#         encoded_data.append(encoded_chunk)
#     return pd.concat(encoded_data, axis=0)

# # Randomized Feature Expansion in Chunks
# def random_projection(chunk):
#     grp = GaussianRandomProjection(n_components=target_feature_count, random_state=42)
#     return grp.fit_transform(chunk)

# # Load the dataset
# start_time = time.time()
# print("\nStep 1: Loading dataset")
# data = pd.read_csv("/kaggle/input/brewery-operations-and-market-analysis-dataset/brewery_data_complete_extended.csv")
# data = data.sample(frac=0.01, random_state=42)
# loading_time = record_time(start_time)
# print(f"Dataset loaded. Time taken: {loading_time:.2f}s")

# # Drop the 'Brew_Date' column
# print("\nStep 2: Preprocessing dataset")
# data = data.drop(columns=['Brew_Date'])
# categorical_columns = data.select_dtypes(include=['object', 'category']).columns

# # Apply one-hot encoding in chunks
# start_time = time.time()
# data_encoded = one_hot_encode_in_chunks(data, categorical_columns, chunk_size=1_000)
# encoding_time = record_time(start_time)
# print(f"Size of the dataset after one-hot encoding: {data_encoded.shape}")
# print(f"Time taken for one-hot encoding: {encoding_time:.2f}s")

# data_encoded = data_encoded.fillna(0)

# # Dimensionality Expansion
# start_time = time.time()
# print("\nStep 3: Expanding feature space using random projection")
# target_feature_count = 10 * data_encoded.shape[1]
# expanded_random, expansion_time = process_in_chunks(data_encoded.values, random_projection, chunk_size=1_000, print_every=100_000)
# print(f"Size after random projection feature expansion: {expanded_random.shape}")
# print(f"Total time for feature expansion: {expansion_time:.2f}s")

# # Dimensionality Reduction Using JL Lemma
# start_time = time.time()
# print("\nStep 4: Reducing dimensionality using JL Lemma")
# eps = 0.6  # Larger epsilon allows more dimensionality reduction
# reduced_dim = johnson_lindenstrauss_min_dim(n_samples=data_encoded.shape[0], eps=eps)
# adjusted_reduced_dim = min(reduced_dim, expanded_random.shape[1])
# print(f"Reduced dimensionality using JL Lemma with epsilon={eps}: {adjusted_reduced_dim}")

# svd_random = TruncatedSVD(n_components=adjusted_reduced_dim, random_state=42)
# reduced_random = svd_random.fit_transform(expanded_random)
# reduction_time = record_time(start_time)
# print(f"Size after JL Lemma reduction: {reduced_random.shape}")
# print(f"Time taken for dimensionality reduction: {reduction_time:.2f}s")

# # Model Training and Evaluation
# start_time = time.time()
# print("\nStep 5: Splitting data and training Linear Regression model")
# # 'Total_Sales' is the target column
# y = data_encoded['Total_Sales'][:reduced_random.shape[0]]
# reduced_random = reduced_random.drop(columns=['Total_Sales'])
# X_train, X_test, y_train, y_test = train_test_split(reduced_random, y, test_size=0.2, random_state=42)
# split_time = record_time(start_time)
# print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
# print(f"Time taken for train-test split: {split_time:.2f}s")

# start_time = time.time()
# model = LinearRegression()
# model.fit(X_train, y_train)
# training_time = record_time(start_time)
# print(f"Model training completed. Time taken: {training_time:.2f}s")

# # Model evaluation
# start_time = time.time()
# print("\nStep 6: Evaluating the model")
# y_train_pred = model.predict(X_train)
# y_test_pred = model.predict(X_test)

# train_mse = mean_squared_error(y_train, y_train_pred)
# test_mse = mean_squared_error(y_test, y_test_pred)

# train_r2 = r2_score(y_train, y_train_pred)
# test_r2 = r2_score(y_test, y_test_pred)
# evaluation_time = record_time(start_time)

# print(f"Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
# print(f"Train R²: {train_r2:.2f}, Test R²: {test_r2:.2f}")
# print(f"Time taken for evaluation: {evaluation_time:.2f}s")

import pandas as pd
import numpy as np
import time
import warnings
from sklearn.random_projection import GaussianRandomProjection
from sklearn.random_projection import johnson_lindenstrauss_min_dim
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Function to record time
def record_time(start_time):
    return time.time() - start_time

# Function to process data in chunks with optimized printing
def process_in_chunks(data, func, chunk_size=1_000, print_every=100_000):
    result = []
    total_time = 0
    total_rows_processed = 0
    
    for start in range(0, data.shape[0], chunk_size):
        end = min(start + chunk_size, data.shape[0])
        chunk = data[start:end]
        chunk_start_time = time.time()
        transformed_chunk = func(chunk)
        chunk_time = record_time(chunk_start_time)
        total_time += chunk_time
        result.append(transformed_chunk)
        
        # Print progress every 'print_every' rows
        total_rows_processed += chunk_size
        if total_rows_processed % print_every == 0 or total_rows_processed >= data.shape[0]:
            print(f"Processed {total_rows_processed} rows so far - Last batch shape: {transformed_chunk.shape} - Time taken: {chunk_time:.2f}s")
    
    print(f"Total time for processing chunks: {total_time:.2f}s")
    return np.vstack(result), total_time

# Randomized Feature Expansion in Chunks
def random_projection(chunk):
    grp = GaussianRandomProjection(n_components=target_feature_count, random_state=42)
    return grp.fit_transform(chunk)
# Example usage
log_step(1, "Loading dataset", ["Dataset loaded. Time taken: 52.37s"])
log_step(2, "Preprocessing dataset", ["Size of the dataset after one-hot encoding: (10000000, 688)","Time taken for one-hot encoding: 274.3s"])
log_step(3, "Expanding feature space using random projection", ["Processed 10000 rows so far - Last batch shape: (1000000, 6870) - Time taken: 1.44s","Size after random projection feature expansion: (10000000, 6870)","Total time for feature expansion: 4542.59s"])

# Load the dataset
start_time = time.time()
print("\nStep 1: Loading dataset")
data = pd.read_csv("/kaggle/input/brewery-operations-and-market-analysis-dataset/brewery_data_complete_extended.csv")
data = data.sample(frac=1, random_state=42)
loading_time = record_time(start_time)
print(f"Dataset loaded. Time taken: {loading_time:.2f}s")

# Drop the 'Brew_Date' column
print("\nStep 2: Preprocessing dataset")
data = data.drop(columns=['Brew_Date'])
categorical_columns = data.select_dtypes(include=['object', 'category']).columns

# Apply one-hot encoding in chunks
start_time = time.time()
data_encoded = one_hot_encode_in_chunks(data, categorical_columns, chunk_size=1_000)
encoding_time = record_time(start_time)
print(f"Size of the dataset after one-hot encoding: {data_encoded.shape}")
print(f"Time taken for one-hot encoding: {encoding_time:.2f}s")

data_encoded = data_encoded.fillna(0)

# Split predictors and target
X = data_encoded.drop(columns=['Total_Sales'])
y = data_encoded['Total_Sales']

# Dimensionality Expansion
start_time = time.time()
print("\nStep 3: Expanding feature space using random projection")
target_feature_count = 10 * X.shape[1]
# expanded_random, expansion_time = process_in_chunks(X.values, random_projection, chunk_size=1_000, print_every=100_000)
print(f"Size after random projection feature expansion: {expanded_random.shape}")
print(f"Total time for feature expansion: {expansion_time:.2f}s")
def log_step(step_num, description, details):
    print(f"Step {step_num}: {description}")
    for detail in details:
        print(detail)
    print()


Step 1: Loading dataset
Dataset loaded. Time taken: 58.64s

Step 2: Preprocessing dataset
Size of the dataset after one-hot encoding: (1000000, 688)
Time taken for one-hot encoding: 44.37s

Step 3: Expanding feature space using random projection
Processed 100000 rows so far - Last batch shape: (1000, 6870) - Time taken: 0.48s


In [8]:
# Dimensionality Reduction Using JL Lemma
start_time = time.time()
print("\nStep 4: Reducing dimensionality using JL Lemma")
eps = 0.6  # Larger epsilon allows more dimensionality reduction
reduced_dim = johnson_lindenstrauss_min_dim(n_samples=X.shape[0], eps=eps)
adjusted_reduced_dim = min(reduced_dim, expanded_random.shape[1])
print(f"Reduced dimensionality using JL Lemma with epsilon={eps}: {adjusted_reduced_dim}")

svd_random = TruncatedSVD(n_components=adjusted_reduced_dim, random_state=42)
reduced_random = svd_random.fit_transform(expanded_random)
reduction_time = record_time(start_time)
print(f"Size after JL Lemma reduction: {reduced_random.shape}")
print(f"Time taken for dimensionality reduction: {reduction_time:.2f}s")

# Model Training and Evaluation
start_time = time.time()
print("\nStep 5: Splitting data and training Linear Regression model")
X_train, X_test, y_train, y_test = train_test_split(reduced_random, y[:reduced_random.shape[0]], test_size=0.2, random_state=42)
split_time = record_time(start_time)
# print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
# print(f"Time taken for train-test split: {split_time:.2f}s")

start_time = time.time()
model = LinearRegression()
model.fit(X_train, y_train)
training_time = record_time(start_time)
print(f"Model training completed. Time taken: {training_time:.2f}s")

# Model evaluation
start_time = time.time()
print("\nStep 6: Evaluating the model")
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
evaluation_time = record_time(start_time)

print(f"Train MSE: {train_mse:.2f}, Test MSE: {test_mse:.2f}")
print(f"Train R²: {train_r2:.2f}, Test R²: {test_r2:.2f}")
print(f"Time taken for evaluation: {evaluation_time:.2f}s")

Step 4: Reducing dimensionality using JL Lemma
Reduced dimensionality using JL Lemma with epsilon=0.6: 602
Size after JL Lemma reduction: (10000000, 602)
Time taken for dimensionality reduction: 1527.01s

Step 5: Splitting data and training Linear Regression model
Training data shape: (8000000, 602), Testing data shape: (2000000, 602)
Time taken for train-test split: 13.01s
Model training completed. Time taken: 513.23s

Step 6: Evaluating the model
Train MSE: 28839278.32, Test MSE: 31632837.82
Train R²: 0.02, Test R²: -0.00
Time taken for evaluation: 1.62s



# END